# 🥗 Sistem Klasifikasi Status Gizi Anak
**Capstone Coding Camp 2026**

Notebook ini berisi pipeline lengkap:
1. ✅ Install & Import
2. ✅ Generate Dataset Sintetis
3. ✅ Custom Layer, Custom Loss, Custom Callback
4. ✅ Bangun Model (TensorFlow Functional API)
5. ✅ Training Model
6. ✅ Evaluasi & Visualisasi
7. ✅ Simpan Model (`.keras` — siap produksi)
8. ✅ Inference (prediksi satu anak & batch)
9. ✅ Demo Flask API (ngrok tunnel)

> **💡 Tips:** Aktifkan GPU di Colab → `Runtime > Change runtime type > T4 GPU`

---
## 1. Install & Import

In [ ]:
# Install library tambahan (scikit-learn sudah tersedia di Colab)
!pip install -q pyngrok flask

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow versi : {tf.__version__}')
print(f'GPU tersedia     : {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## 2. Dataset

Dataset berisi fitur antropometri anak:
| Fitur | Keterangan |
|---|---|
| `umur_bulan` | Usia anak dalam bulan (0–60) |
| `berat_badan_kg` | Berat badan (kg) |
| `tinggi_badan_cm` | Tinggi badan (cm) |
| `jenis_kelamin` | 0 = Laki-laki, 1 = Perempuan |
| `lingkar_lengan` | Lingkar lengan atas (cm) |
| `lingkar_kepala` | Lingkar kepala (cm) |

**Label:** `0`=Gizi Buruk, `1`=Gizi Kurang, `2`=Gizi Baik, `3`=Gizi Lebih, `4`=Obesitas

> 🔄 **Ganti dengan data nyata** dari tim Data Science: cukup ubah fungsi `load_data()` di bawah untuk membaca CSV Anda.

In [ ]:
CLASS_NAMES  = ['Gizi Buruk', 'Gizi Kurang', 'Gizi Baik', 'Gizi Lebih', 'Obesitas']
NUM_CLASSES  = len(CLASS_NAMES)
INPUT_DIM    = 6
FEATURE_COLS = ['umur_bulan', 'berat_badan_kg', 'tinggi_badan_cm',
                'jenis_kelamin', 'lingkar_lengan', 'lingkar_kepala']

def generate_synthetic_data(n_samples=3000):
    """Dataset sintetis — ganti fungsi ini dengan loader CSV nyata."""
    np.random.seed(SEED)
    class_probs = [0.05, 0.15, 0.55, 0.15, 0.10]  # distribusi kelas tidak seimbang
    labels = np.random.choice(NUM_CLASSES, size=n_samples, p=class_probs)

    rows = []
    for label in labels:
        umur    = np.random.randint(0, 60)
        kelamin = np.random.randint(0, 2)
        berat_mean  = [8,  11, 14, 18, 22][label]
        tinggi_mean = [65, 75, 85, 90, 92][label]
        lengan_mean = [11, 12, 14, 16, 18][label]
        kepala_mean = [44, 46, 48, 50, 51][label]
        rows.append([
            umur,
            np.clip(np.random.normal(berat_mean,  1.5), 2, 35),
            np.clip(np.random.normal(tinggi_mean, 5.0), 40, 120),
            kelamin,
            np.clip(np.random.normal(lengan_mean, 1.0), 8, 25),
            np.clip(np.random.normal(kepala_mean, 1.0), 30, 58),
        ])

    df = pd.DataFrame(rows, columns=FEATURE_COLS)
    df['label'] = labels
    return df

# ── Untuk load data nyata, ganti dengan:
# df = pd.read_csv('data_gizi.csv')

df = generate_synthetic_data(3000)
print(f'Shape dataset : {df.shape}')
print(f'\nDistribusi kelas:')
for i, name in enumerate(CLASS_NAMES):
    count = (df['label'] == i).sum()
    pct   = count / len(df) * 100
    bar   = '█' * int(pct / 2)
    print(f'  {name:<15}: {count:4d} ({pct:.1f}%)  {bar}')

df.head()

In [ ]:
# Visualisasi distribusi fitur per kelas
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribusi Fitur per Status Gizi', fontsize=14, fontweight='bold')
palette = sns.color_palette('Set2', NUM_CLASSES)

for ax, col in zip(axes.flat, FEATURE_COLS):
    for i, name in enumerate(CLASS_NAMES):
        subset = df[df['label'] == i][col]
        ax.hist(subset, bins=25, alpha=0.6, label=name, color=palette[i])
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=7)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Persiapan data
X = df[FEATURE_COLS].values.astype(np.float32)
y = keras.utils.to_categorical(df['label'].values, num_classes=NUM_CLASSES)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=df['label'])
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f'Train : {X_train.shape[0]} sampel')
print(f'Val   : {X_val.shape[0]} sampel')
print(f'Test  : {X_test.shape[0]} sampel')

---
## 3. Komponen Kustom
### 3a. Custom Layer — `NutritionNormalizationLayer`

In [ ]:
class NutritionNormalizationLayer(keras.layers.Layer):
    """
    Custom Layer: Z-score normalization yang menjadi bagian dari model.
    Keuntungan: saat deploy, tidak perlu scaler terpisah — preprocessing
    sudah baked-in di dalam file .keras.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        n = input_shape[-1]
        self.mean = self.add_weight('mean', shape=(n,), initializer='zeros', trainable=False)
        self.std  = self.add_weight('std',  shape=(n,), initializer='ones',  trainable=False)
        super().build(input_shape)

    def adapt(self, data):
        """Hitung mean & std dari data training — panggil sebelum model.fit()."""
        self.mean.assign(tf.reduce_mean(data, axis=0))
        self.std.assign(tf.math.reduce_std(data, axis=0) + 1e-8)
        print('NutritionNormalizationLayer — adapt selesai:')
        for feat, m, s in zip(FEATURE_COLS, self.mean.numpy(), self.std.numpy()):
            print(f'  {feat:<20}: mean={m:.3f}, std={s:.3f}')

    def call(self, inputs):
        return (inputs - self.mean) / self.std

    def get_config(self):
        return super().get_config()

print('✅ NutritionNormalizationLayer didefinisikan')

### 3b. Custom Loss — `FocalLoss`

In [ ]:
class FocalLoss(keras.losses.Loss):
    """
    Custom Loss: Focal Loss untuk menangani class imbalance.
    Kelas minoritas (misal 'Gizi Buruk') mendapat bobot lebih besar
    sehingga model tidak abai terhadap sampel yang langka.

    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)
    - gamma > 0 → turunkan bobot sampel yang mudah (well-classified)
    - alpha      → keseimbangan positif/negatif
    """

    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight        = self.alpha * tf.pow(1.0 - y_pred, self.gamma)
        focal_loss    = weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=-1))

    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'alpha': self.alpha})
        return config

print('✅ FocalLoss didefinisikan')

### 3c. Custom Callback — `EarlyStoppingWithLogging`

In [ ]:
class EarlyStoppingWithLogging(keras.callbacks.Callback):
    """
    Custom Callback yang:
    1. Early stopping berdasarkan val_loss
    2. Simpan log training ke file .txt setiap epoch
    3. Tampilkan ringkasan performa di akhir training
    """

    def __init__(self, patience=15, log_path='training_log.txt'):
        super().__init__()
        self.patience       = patience
        self.log_path       = log_path
        self.best_val_loss  = np.inf
        self.wait           = 0
        self.best_epoch     = 0
        self.history_rows   = []

    def on_train_begin(self, logs=None):
        self.wait          = 0
        self.best_val_loss = np.inf
        with open(self.log_path, 'w') as f:
            f.write('epoch,loss,accuracy,val_loss,val_accuracy\n')
        print('\n' + '='*65)
        print('  Training Model Klasifikasi Status Gizi Anak')
        print('='*65)

    def on_epoch_end(self, epoch, logs=None):
        logs       = logs or {}
        val_loss   = logs.get('val_loss',     np.inf)
        val_acc    = logs.get('val_accuracy', 0.0)
        train_loss = logs.get('loss',         np.inf)
        train_acc  = logs.get('accuracy',     0.0)

        improved = val_loss < self.best_val_loss
        if improved:
            self.best_val_loss = val_loss
            self.best_epoch    = epoch + 1
            self.wait          = 0
            flag = ' ★ TERBAIK'
        else:
            self.wait += 1
            flag = f' (tidak membaik {self.wait}/{self.patience})'

        print(f'Epoch {epoch+1:03d} | loss={train_loss:.4f} acc={train_acc:.4f} '
              f'| val_loss={val_loss:.4f} val_acc={val_acc:.4f}{flag}')

        with open(self.log_path, 'a') as f:
            f.write(f'{epoch+1},{train_loss:.4f},{train_acc:.4f},'
                    f'{val_loss:.4f},{val_acc:.4f}\n')

        self.history_rows.append({
            'epoch': epoch+1, 'loss': train_loss, 'accuracy': train_acc,
            'val_loss': val_loss, 'val_accuracy': val_acc
        })

        if self.wait >= self.patience:
            self.model.stop_training = True
            print(f'\n  ⛔ Early stopping! Kembali ke epoch terbaik: {self.best_epoch}')

    def on_train_end(self, logs=None):
        print('\n' + '='*65)
        print(f'  ✅ Training selesai  |  Epoch terbaik : {self.best_epoch}')
        print(f'     Best val_loss    : {self.best_val_loss:.4f}')
        print(f'     Log disimpan di  : {self.log_path}')
        print('='*65 + '\n')

print('✅ EarlyStoppingWithLogging didefinisikan')

---
## 4. Bangun Model (Functional API)

In [ ]:
def build_model():
    """
    Arsitektur menggunakan TensorFlow Functional API.
    Input → NormLayer → Dense(128) → BN → Dropout(0.3)
          → Dense(64)  → BN → Dropout(0.2)
          → Dense(32)  → Output(softmax, 5 kelas)
    """
    norm_layer = NutritionNormalizationLayer(name='normalization')

    inputs = keras.Input(shape=(INPUT_DIM,), name='input_gizi')

    # ── Custom normalization layer
    x = norm_layer(inputs)

    # ── Blok 1
    x = layers.Dense(128, activation='relu',       name='dense_1')(x)
    x = layers.BatchNormalization(                  name='bn_1')(x)
    x = layers.Dropout(0.3,                         name='drop_1')(x)

    # ── Blok 2
    x = layers.Dense(64, activation='relu',         name='dense_2')(x)
    x = layers.BatchNormalization(                  name='bn_2')(x)
    x = layers.Dropout(0.2,                         name='drop_2')(x)

    # ── Blok 3
    x = layers.Dense(32, activation='relu',         name='dense_3')(x)

    # ── Output
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='NutritionClassifier')
    return model, norm_layer

model, norm_layer = build_model()
model.summary()

In [ ]:
# Adapt normalization layer dengan data training
norm_layer.adapt(X_train)

# Kompilasi
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=FocalLoss(gamma=2.0, alpha=0.25),
    metrics=['accuracy'],
)
print('\n✅ Model siap dilatih!')

---
## 5. Training

In [ ]:
os.makedirs('saved_model', exist_ok=True)

early_stop = EarlyStoppingWithLogging(patience=15, log_path='training_log.txt')

callbacks = [
    early_stop,
    keras.callbacks.ModelCheckpoint(
        filepath='saved_model/best_checkpoint.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-6, verbose=1,
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=0,    # output ditangani custom callback
)

---
## 6. Evaluasi & Visualisasi

In [ ]:
# ── Kurva training
log_df = pd.DataFrame(early_stop.history_rows)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Kurva Training', fontsize=13, fontweight='bold')

ax1.plot(log_df['epoch'], log_df['loss'],       label='Train Loss', color='#E24B4A')
ax1.plot(log_df['epoch'], log_df['val_loss'],   label='Val Loss',   color='#378ADD', linestyle='--')
ax1.axvline(early_stop.best_epoch, color='gray', linestyle=':', label=f'Best epoch ({early_stop.best_epoch})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Focal Loss'); ax1.legend(); ax1.set_title('Loss')

ax2.plot(log_df['epoch'], log_df['accuracy'],     label='Train Acc', color='#1D9E75')
ax2.plot(log_df['epoch'], log_df['val_accuracy'], label='Val Acc',   color='#BA7517', linestyle='--')
ax2.axvline(early_stop.best_epoch, color='gray', linestyle=':')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_title('Accuracy')

plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluasi pada data test
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss    : {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')

y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test, axis=1)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ── Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1)
ax1.set_title('Confusion Matrix (jumlah)', fontweight='bold')
ax1.set_xlabel('Prediksi'); ax1.set_ylabel('Aktual')

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2)
ax2.set_title('Confusion Matrix (%)', fontweight='bold')
ax2.set_xlabel('Prediksi'); ax2.set_ylabel('Aktual')

plt.tight_layout()
plt.show()

---
## 7. Simpan Model (Format Produksi `.keras`)

In [ ]:
MODEL_PATH = 'saved_model/nutrition_classifier.keras'
model.save(MODEL_PATH)
print(f'✅ Model disimpan ke: {MODEL_PATH}')

# Verifikasi — load ulang dan cek
loaded_model = keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        'NutritionNormalizationLayer': NutritionNormalizationLayer,
        'FocalLoss': FocalLoss,
    }
)
_, loaded_acc = loaded_model.evaluate(X_test, y_test, verbose=0)
print(f'✅ Verifikasi model yang di-load: accuracy = {loaded_acc:.4f}')
print('   (harus sama dengan nilai sebelumnya)')

In [ ]:
# Download model ke komputer lokal
from google.colab import files
files.download(MODEL_PATH)
print('📥 Download dimulai...')

---
## 8. Inference
### 8a. Prediksi Satu Anak

In [ ]:
def predict_single(data: dict, mdl=None) -> dict:
    """
    Prediksi status gizi satu anak dari dictionary input.
    
    Args:
        data: dict dengan key umur_bulan, berat_badan_kg, tinggi_badan_cm,
                          jenis_kelamin, lingkar_lengan, lingkar_kepala
    Returns:
        dict: status_gizi, kelas_id, probabilitas, confidence
    """
    if mdl is None:
        mdl = model   # gunakan model yang sudah dilatih

    required = ['umur_bulan', 'berat_badan_kg', 'tinggi_badan_cm',
                'jenis_kelamin', 'lingkar_lengan', 'lingkar_kepala']
    missing = [k for k in required if k not in data]
    if missing:
        raise ValueError(f'Kunci input tidak lengkap: {missing}')

    features = np.array([[data[k] for k in required]], dtype=np.float32)
    proba    = mdl.predict(features, verbose=0)[0]
    kelas_id = int(np.argmax(proba))

    return {
        'status_gizi' : CLASS_NAMES[kelas_id],
        'kelas_id'    : kelas_id,
        'probabilitas': {c: round(float(p), 4) for c, p in zip(CLASS_NAMES, proba)},
        'confidence'  : round(float(proba[kelas_id]), 4),
    }

# ── Demo
contoh_anak = {
    'umur_bulan'     : 24,
    'berat_badan_kg' : 11.5,
    'tinggi_badan_cm': 84.0,
    'jenis_kelamin'  : 0,       # 0=Laki-laki
    'lingkar_lengan' : 14.2,
    'lingkar_kepala' : 47.5,
}

hasil = predict_single(contoh_anak)

print('=== Hasil Prediksi ===')
print(f"Status Gizi : {hasil['status_gizi']}")
print(f"Confidence  : {hasil['confidence']:.2%}")
print('\nProbabilitas per kelas:')
for cls, prob in hasil['probabilitas'].items():
    bar = '█' * int(prob * 40)
    print(f'  {cls:<15}: {prob:.4f}  {bar}')

### 8b. Prediksi Batch

In [ ]:
def predict_batch(data_list: list, mdl=None) -> pd.DataFrame:
    """Prediksi untuk banyak anak sekaligus, kembalikan sebagai DataFrame."""
    if mdl is None:
        mdl = model
    required = ['umur_bulan', 'berat_badan_kg', 'tinggi_badan_cm',
                'jenis_kelamin', 'lingkar_lengan', 'lingkar_kepala']
    features = np.array([[d[k] for k in required] for d in data_list], dtype=np.float32)
    probas   = mdl.predict(features, verbose=0)
    rows = []
    for i, (d, proba) in enumerate(zip(data_list, probas)):
        kelas_id = int(np.argmax(proba))
        rows.append({
            **d,
            'status_gizi': CLASS_NAMES[kelas_id],
            'confidence' : round(float(proba[kelas_id]), 4),
        })
    return pd.DataFrame(rows)

# ── Demo batch 5 anak
batch_input = [
    {'umur_bulan':  6, 'berat_badan_kg': 5.2,  'tinggi_badan_cm': 62.0, 'jenis_kelamin': 1, 'lingkar_lengan': 10.5, 'lingkar_kepala': 41.0},
    {'umur_bulan': 12, 'berat_badan_kg': 9.5,  'tinggi_badan_cm': 75.0, 'jenis_kelamin': 0, 'lingkar_lengan': 13.0, 'lingkar_kepala': 46.0},
    {'umur_bulan': 24, 'berat_badan_kg': 11.5, 'tinggi_badan_cm': 84.0, 'jenis_kelamin': 0, 'lingkar_lengan': 14.2, 'lingkar_kepala': 47.5},
    {'umur_bulan': 36, 'berat_badan_kg': 18.0, 'tinggi_badan_cm': 91.0, 'jenis_kelamin': 1, 'lingkar_lengan': 16.5, 'lingkar_kepala': 50.0},
    {'umur_bulan': 48, 'berat_badan_kg': 23.5, 'tinggi_badan_cm': 94.0, 'jenis_kelamin': 0, 'lingkar_lengan': 18.2, 'lingkar_kepala': 51.5},
]

hasil_batch = predict_batch(batch_input)
print('=== Hasil Prediksi Batch ===')
display(hasil_batch[['umur_bulan', 'berat_badan_kg', 'tinggi_badan_cm', 'status_gizi', 'confidence']])

---
## 9. Demo Flask API dengan ngrok

> **Opsional** — jalankan cell ini untuk mengekspos API ke internet via ngrok.
> 
> Daftar akun gratis di https://ngrok.com → salin Auth Token → paste di bawah.

In [ ]:
# ── Tulis app.py ke disk
app_code = '''
import numpy as np
import tensorflow as tf
from tensorflow import keras
from flask import Flask, request, jsonify

CLASS_NAMES  = ["Gizi Buruk", "Gizi Kurang", "Gizi Baik", "Gizi Lebih", "Obesitas"]
FEATURE_COLS = ["umur_bulan", "berat_badan_kg", "tinggi_badan_cm",
                "jenis_kelamin", "lingkar_lengan", "lingkar_kepala"]

app   = Flask(__name__)
_model = None

def get_model():
    global _model
    if _model is None:
        from model_classes import NutritionNormalizationLayer, FocalLoss
        _model = keras.models.load_model(
            "saved_model/nutrition_classifier.keras",
            custom_objects={"NutritionNormalizationLayer": NutritionNormalizationLayer,
                            "FocalLoss": FocalLoss}
        )
    return _model

@app.route("/")
def index():
    return jsonify({"service": "Klasifikasi Gizi Anak", "status": "running"})

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(force=True)
    try:
        features = np.array([[data[k] for k in FEATURE_COLS]], dtype=np.float32)
        proba    = get_model().predict(features, verbose=0)[0]
        kelas_id = int(np.argmax(proba))
        return jsonify({
            "status_gizi" : CLASS_NAMES[kelas_id],
            "kelas_id"    : kelas_id,
            "confidence"  : round(float(proba[kelas_id]), 4),
            "probabilitas": {c: round(float(p), 4) for c, p in zip(CLASS_NAMES, proba)},
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 422

if __name__ == "__main__":
    app.run(port=5000)
'''

with open('flask_app.py', 'w') as f:
    f.write(app_code)
print('✅ flask_app.py ditulis')

In [ ]:
# Konfigurasi ngrok — masukkan token Anda
NGROK_TOKEN = 'ISI_TOKEN_NGROK_ANDA_DI_SINI'  # ← ganti ini

if NGROK_TOKEN != 'ISI_TOKEN_NGROK_ANDA_DI_SINI':
    import threading
    from pyngrok import ngrok, conf

    # Set token
    conf.get_default().auth_token = NGROK_TOKEN

    # Import custom objects agar tersedia saat Flask load model
    import sys, types
    mod = types.ModuleType('model_classes')
    mod.NutritionNormalizationLayer = NutritionNormalizationLayer
    mod.FocalLoss = FocalLoss
    sys.modules['model_classes'] = mod

    # Jalankan Flask di thread terpisah
    import importlib, flask_app as fa
    flask_thread = threading.Thread(target=lambda: fa.app.run(port=5000), daemon=True)
    flask_thread.start()

    # Buka tunnel
    public_url = ngrok.connect(5000).public_url
    print(f'\n🌐 API publik tersedia di: {public_url}')
    print(f'   Endpoint prediksi     : {public_url}/predict')
    print('\nContoh curl:')
    print(f'''  curl -X POST {public_url}/predict \\
    -H "Content-Type: application/json" \\
    -d \'{{"umur_bulan":24,"berat_badan_kg":11.5,"tinggi_badan_cm":84,"jenis_kelamin":0,"lingkar_lengan":14.2,"lingkar_kepala":47.5}}\'\n''')
else:
    print('⚠️  Masukkan NGROK_TOKEN Anda terlebih dahulu.')
    print('   Daftar gratis di https://ngrok.com')

---
## 10. Ringkasan Proyek

| Komponen | Implementasi |
|---|---|
| **Arsitektur** | TensorFlow Functional API |
| **Custom Layer** | `NutritionNormalizationLayer` — Z-score normalization baked-in |
| **Custom Loss** | `FocalLoss(γ=2, α=0.25)` — menangani class imbalance |
| **Custom Callback** | `EarlyStoppingWithLogging` — early stop + log CSV |
| **Format Simpan** | `.keras` (TF siap produksi) |
| **Inference** | Fungsi `predict_single()` & `predict_batch()` |
| **API** | Flask REST (`/predict`, `/predict/batch`) + ngrok tunnel |

### Langkah Selanjutnya
1. **Ganti data sintetis** → load CSV nyata dari tim Data Science
2. **Hyperparameter tuning** → coba `gamma`, `alpha`, `learning_rate`, ukuran layer
3. **Deploy** → export model ke Flask server / Google Cloud Run / Hugging Face Spaces